# Update Heisig from Japanese Media (AnkiConnect)

Workflow:
1. Read review notes from `Japanese Media` deck.
2. Extract kanji from `Lemma`.
3. Find matching notes in `Japanese Heisig` by `Kanji` field.
4. Unsuspend + tag matched Heisig notes.
5. Ensure `Link` field exists on all note types in both decks.
6. Write bidirectional links as `[Title|nid123...]` into `Link`.

Run once with `DRY_RUN = True`, then switch to `False` to apply.

In [1]:
import json
import re
import urllib.request
from collections import defaultdict

ANKI_CONNECT_URL = "http://127.0.0.1:8765"
ANKI_CONNECT_VERSION = 6

MEDIA_DECK = "Japanese Media"
HEISIG_DECK = "Japanese Heisig"
MEDIA_FIELD = "Lemma"
HEISIG_FIELD = "Kanji"
LINK_FIELD = "Link"
TAG = "currently_learning"
DRY_RUN = True

KANJI_RE = re.compile(r"[\u4e00-\u9fff]")

def invoke(action, **params):
    payload = json.dumps({"action": action, "version": ANKI_CONNECT_VERSION, "params": params}).encode("utf-8")
    req = urllib.request.Request(ANKI_CONNECT_URL, data=payload, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req) as resp:
        body = json.loads(resp.read().decode("utf-8"))
    if body.get("error"):
        raise RuntimeError(f"AnkiConnect error in {action}: {body['error']}")
    return body.get("result")

def chunked(items, size=500):
    for i in range(0, len(items), size):
        yield items[i:i + size]

def get_notes_info(note_ids):
    out = []
    for batch in chunked(note_ids, 500):
        out.extend(invoke("notesInfo", notes=batch))
    return out

def sanitize_title(text):
    # Keep links readable and avoid breaking bracket/pipe syntax.
    text = (text or "").replace("[", "(").replace("]", ")").replace("|", "/")
    return text.strip()

def make_link(title, nid):
    return f"[{sanitize_title(title)}|nid{nid}]"

print("Connected AnkiConnect version:", invoke("version"))

Connected AnkiConnect version: 6


In [2]:
# 1) Read review notes from Japanese Media and extract kanji from Lemma
media_query = f'deck:"{MEDIA_DECK}" is:review'
media_note_ids = invoke("findNotes", query=media_query)
media_infos = get_notes_info(media_note_ids)

media_note_kanji = {}  # nid -> set(kanji)
media_note_lemma = {}  # nid -> lemma text
all_kanji = set()
missing_media_field = 0

for info in media_infos:
    nid = info["noteId"]
    fields = info.get("fields", {})
    if MEDIA_FIELD not in fields:
        missing_media_field += 1
        continue

    lemma = fields[MEDIA_FIELD].get("value", "") or ""
    ks = set(KANJI_RE.findall(lemma))
    media_note_kanji[nid] = ks
    media_note_lemma[nid] = lemma
    all_kanji.update(ks)

print({
    "media_review_notes_found": len(media_note_ids),
    "media_notes_with_lemma_field": len(media_note_kanji),
    "media_notes_missing_lemma_field": missing_media_field,
    "unique_kanji_extracted": len(all_kanji),
})

{'media_review_notes_found': 867, 'media_notes_with_lemma_field': 867, 'media_notes_missing_lemma_field': 0, 'unique_kanji_extracted': 734}


In [3]:
# 2) Find matching Heisig notes by Kanji field
heisig_query = f'deck:"{HEISIG_DECK}"'
heisig_note_ids = invoke("findNotes", query=heisig_query)
heisig_infos = get_notes_info(heisig_note_ids)

heisig_note_kanji = {}  # heisig nid -> kanji chars contained in Kanji field
kanji_to_heisig_nids = defaultdict(set)
matching_heisig_nids = set()
heisig_card_ids_to_unsuspend = []
missing_heisig_field = 0

for info in heisig_infos:
    nid = info["noteId"]
    fields = info.get("fields", {})
    if HEISIG_FIELD not in fields:
        missing_heisig_field += 1
        continue

    kanji_val = fields[HEISIG_FIELD].get("value", "") or ""
    ks = set(KANJI_RE.findall(kanji_val))
    heisig_note_kanji[nid] = ks

    shared = ks & all_kanji
    if shared:
        matching_heisig_nids.add(nid)
        heisig_card_ids_to_unsuspend.extend(info.get("cards", []))
        for k in shared:
            kanji_to_heisig_nids[k].add(nid)

# 3) Ensure Link field exists on note types used in both decks
all_infos = media_infos + heisig_infos
model_names = sorted({n.get("modelName") for n in all_infos if n.get("modelName")})
added_link_field_models = []

for model in model_names:
    existing_fields = invoke("modelFieldNames", modelName=model)
    if LINK_FIELD not in existing_fields:
        if not DRY_RUN:
            invoke("modelFieldAdd", modelName=model, fieldName=LINK_FIELD)
        added_link_field_models.append(model)

# Build reverse map: kanji -> media nids
kanji_to_media_nids = defaultdict(set)
for mnid, ks in media_note_kanji.items():
    for k in ks:
        if k in all_kanji:
            kanji_to_media_nids[k].add(mnid)

# 4) Build and write Link field updates
heisig_updates = []
for hnid in matching_heisig_nids:
    ks = heisig_note_kanji.get(hnid, set())
    linked_media = set()
    for k in ks:
        linked_media.update(kanji_to_media_nids.get(k, set()))

    if not linked_media:
        continue

    links = [make_link(f"Media {mnid}", mnid) for mnid in sorted(linked_media)]
    heisig_updates.append({"id": hnid, "fields": {LINK_FIELD: "<br>".join(links)}})

media_updates = []
for mnid, ks in media_note_kanji.items():
    linked_heisig = set()
    for k in ks:
        linked_heisig.update(kanji_to_heisig_nids.get(k, set()))

    if not linked_heisig:
        continue

    links = [make_link(f"Kanji {k}", hnid) for hnid in sorted(linked_heisig) for k in sorted(heisig_note_kanji.get(hnid, set()) & ks)]
    dedup = list(dict.fromkeys(links))
    media_updates.append({"id": mnid, "fields": {LINK_FIELD: "<br>".join(dedup)}})

if not DRY_RUN:
    if heisig_card_ids_to_unsuspend:
        for batch in chunked(sorted(set(heisig_card_ids_to_unsuspend)), 500):
            invoke("unsuspend", cards=batch)

    if matching_heisig_nids:
        for batch in chunked(sorted(matching_heisig_nids), 500):
            invoke("addTags", notes=batch, tags=TAG)

    for upd in heisig_updates:
        invoke("updateNoteFields", note=upd)

    for upd in media_updates:
        invoke("updateNoteFields", note=upd)

print({
    "dry_run": DRY_RUN,
    "heisig_notes_found": len(heisig_note_ids),
    "heisig_notes_missing_kanji_field": missing_heisig_field,
    "heisig_notes_matched_by_media_kanji": len(matching_heisig_nids),
    "heisig_cards_to_unsuspend": len(set(heisig_card_ids_to_unsuspend)),
    "tag_to_apply": TAG,
    "note_types_checked_for_link_field": len(model_names),
    "note_types_missing_link_field": added_link_field_models,
    "heisig_link_updates": len(heisig_updates),
    "media_link_updates": len(media_updates),
})

if DRY_RUN:
    print("DRY_RUN=True, no changes written.")

{'dry_run': True, 'heisig_notes_found': 8508, 'heisig_notes_missing_kanji_field': 0, 'heisig_notes_matched_by_media_kanji': 2166, 'heisig_cards_to_unsuspend': 2166, 'tag_to_apply': 'currently_learning', 'note_types_checked_for_link_field': 5, 'note_types_missing_link_field': ['HeisigKanjiJapanese', 'Kanji - Heisig order', 'Kanji_RTK_13', 'Moritz Language Reactor', 'Moritz Language Reactor Phrase'], 'heisig_link_updates': 2166, 'media_link_updates': 727}
DRY_RUN=True, no changes written.


## Notes

- This script overwrites the `Link` field each run.
- If you already use `Link` for other content, merge logic should be added before applying.
- Make an Anki backup before running with `DRY_RUN = False`.